# SignalCheck · RoBERTa on prepared data

This notebook trains and evaluates a classifier. **Sample mode** uses a tiny randomly initialized RoBERTa network to test every step offline. It is not a valid news detector. Full mode fine-tunes pretrained `roberta-base` and requires a CUDA GPU and model download.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, split_data, train_roberta, evaluate, save_results

## 1 · Choose data

For a full run, prepare `data/processed/final_dataset.csv` with the script described in the README.

In [4]:
DATA_MODE = "sample"  # change to "processed" for the full dataset
frame = load_data(DATA_MODE)
print(f"Rows: {len(frame):,}; labels: {frame.label.value_counts().to_dict()}")
display(frame[["combined", "label"]].head(3))

Rows: 32; labels: {1: 16, 0: 16}
                                                                                         combined  label
Demo City opens a library The fictional city council opened a new library in the sample district.      1
    Demo City repairs a bridge The fictional city council completed repairs to the sample bridge.      1
   Demo City adds bus stops The fictional transit office added three bus stops to a sample route.      1


## 2 · Stratified split

The text overlap check in `split_data` raises an error if a duplicate crosses a split.

In [6]:
train, valid, test = split_data(frame, sample=DATA_MODE == "sample")
print("Train / validation / test:", len(train), len(valid), len(test))

Train / validation / test: 19 6 7


## 3 · Fine-tune

Sample mode is a CPU smoke test. Full mode uses three epochs of pretrained RoBERTa on CUDA; training time depends on the GPU.

In [8]:
model, tokenizer, history, test_pred, probabilities = train_roberta(
    train, valid, test, sample=DATA_MODE == "sample"
)
print("Training history:", history)

Training history: [{'epoch': 1, 'train_loss': 0.6970609068870545, 'validation_macro_f1': 0.3333333333333333}]


## 4 · Evaluate and save

The test metrics are from this run. Synthetic sample scores do not represent real news performance.

In [10]:
metrics = evaluate(test["label"], test_pred)
print(metrics["classification_report"])
print("Confusion matrix [fake, real]:", metrics["confusion_matrix"])
scope = "sample" if DATA_MODE == "sample" else "full"
output_dir = save_results("roberta_prepared", scope, test["label"], test_pred, probabilities)
model_dir = ROOT / "models" / scope / "roberta_prepared"
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print("Results:", output_dir)
print("Model:", model_dir)

              precision    recall  f1-score   support

        fake       0.57      1.00      0.73         4
        real       0.00      0.00      0.00         3

    accuracy                           0.57         7
   macro avg       0.29      0.50      0.36         7
weighted avg       0.33      0.57      0.42         7

Confusion matrix [fake, real]: [[4, 0], [3, 0]]
Writing model shards: 100%|############################################################################################################| 1/1 [00:00<00:00, 224.21it/s]
Results: C:\Users\Mohd Shoeb\OneDrive\Desktop\Fake-News-Detection-System\results\sample\roberta_prepared
Model: C:\Users\Mohd Shoeb\OneDrive\Desktop\Fake-News-Detection-System\models\sample\roberta_prepared
